# 📱 Mobile Price Classification – Data Analysis
**Dataset:** `dailytrain.csv` | 2 000 smartphones, 20 features + 1 target (`price_range` 0-3)

### Roadmap
1. Data Loading & Exploration  
2. Data Cleaning & Preprocessing  
3. Statistical Analysis (NumPy / SciPy)  
4. Data Visualization (Matplotlib)  
5. Insight Synthesis & Conclusion


## 1. Data Loading & Exploration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import shapiro, f_oneway, pearsonr, spearmanr, skew, kurtosis
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False, 'font.size': 11})
PALETTE = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']


In [ ]:
# Load the dataset
df = pd.read_csv('dailytrain.csv')
print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
df.head()


In [ ]:
# Data types and target distribution
print("=== Data Types ===")
print(df.dtypes)
print()
print("=== Target Distribution (price_range) ===")
print(df['price_range'].value_counts().sort_index())


In [ ]:
# Basic descriptive statistics
# count, mean, std, min, quartiles, max for all numeric columns
df.describe().T


**Observations:**
- Dataset is perfectly **balanced** – exactly 500 devices per price class (0,1,2,3).
- `ram` has the widest range (256–3998 MB) and is likely the strongest predictor.
- Binary columns (blue, dual_sim, four_g, three_g, touch_screen, wifi) are already encoded as 0/1.


## 2. Data Cleaning & Preprocessing

In [ ]:
# Check for missing values
missing = df.isnull().sum()
print("Missing values per column:")
print(missing[missing > 0] if missing.any() else "No missing values found.")


In [ ]:
# Check for duplicate rows
dupes = df.duplicated().sum()
print(f"Duplicate rows: {dupes}")
if dupes:
    df = df.drop_duplicates()
    print("Duplicates removed.")


In [ ]:
# Classify features
binary_cols     = ['blue', 'dual_sim', 'four_g', 'three_g', 'touch_screen', 'wifi']
continuous_cols = [c for c in df.columns if c not in binary_cols + ['price_range']]

print("Binary features   :", binary_cols)
print("Continuous features:", continuous_cols)
print("Target            : price_range")

# All features are already numeric – no label encoding required.
# StandardScaler applied for correlation analysis reference.
from sklearn.preprocessing import StandardScaler
df_scaled = df.copy()
df_scaled[continuous_cols] = StandardScaler().fit_transform(df[continuous_cols])
print("StandardScaler applied to continuous features in df_scaled.")


## 3. Statistical Analysis with NumPy & SciPy

### 3.1 Central Tendency, Variability & Shape

In [ ]:
# Summary table: mean, median, mode, range, variance, std, skewness, kurtosis
rows = []
for col in continuous_cols:
    x = df[col].values
    mode_res = stats.mode(x, keepdims=True)
    rows.append({
        'Feature'  : col,
        'Mean'     : round(np.mean(x), 3),
        'Median'   : round(np.median(x), 3),
        'Mode'     : round(mode_res.mode[0], 3),
        'Range'    : round(np.ptp(x), 3),
        'Variance' : round(np.var(x, ddof=1), 3),
        'Std Dev'  : round(np.std(x, ddof=1), 3),
        'Skewness' : round(skew(x), 3),
        'Kurtosis' : round(kurtosis(x), 3),
    })

stat_df = pd.DataFrame(rows).set_index('Feature')
stat_df


**Reading the table:**
- |Skewness| > 1 → notably asymmetric distribution.
- Kurtosis > 0 (leptokurtic) → heavier tails than Normal; < 0 (platykurtic) → lighter tails.
- `px_height` shows the largest positive skew (long right tail).


### 3.2 Normality Test (Shapiro-Wilk on 500-sample)

In [ ]:
# Shapiro-Wilk reliable up to n~5000; sample 500 for speed
np.random.seed(42)
sample_idx = np.random.choice(len(df), 500, replace=False)

normality = []
for col in continuous_cols:
    stat, p = shapiro(df[col].iloc[sample_idx])
    normality.append({
        'Feature'      : col,
        'W-stat'       : round(stat, 4),
        'p-value'      : round(p, 6),
        'Normal (a=0.05)': 'YES' if p > 0.05 else 'NO'
    })

pd.DataFrame(normality).set_index('Feature')


### 3.3 One-Way ANOVA – Feature Means Across Price Classes

In [ ]:
# H0: means are equal across all 4 price groups
groups = [df[df['price_range'] == g] for g in [0,1,2,3]]

anova_results = []
for col in continuous_cols:
    samples = [g[col].values for g in groups]
    f_stat, p_val = f_oneway(*samples)
    anova_results.append({
        'Feature'      : col,
        'F-statistic'  : round(f_stat, 3),
        'p-value'      : round(p_val, 6),
        'Significant'  : 'YES' if p_val < 0.05 else 'NO'
    })

anova_df = (pd.DataFrame(anova_results)
              .set_index('Feature')
              .sort_values('F-statistic', ascending=False))
anova_df


**Interpretation:** A high F-statistic (p < 0.05) means the feature mean differs significantly across price classes. Features with large F values are the most discriminating.


### 3.4 Correlation with Target (Pearson & Spearman)

In [ ]:
# Pearson: linear correlation. Spearman: monotonic (rank-based) correlation.
corr_results = []
for col in df.columns.drop('price_range'):
    pr, pp = pearsonr(df[col], df['price_range'])
    sr, sp = spearmanr(df[col], df['price_range'])
    corr_results.append({
        'Feature'    : col,
        'Pearson r'  : round(pr, 4),
        'Pearson p'  : round(pp, 6),
        'Spearman rho': round(sr, 4),
        'Spearman p' : round(sp, 6),
    })

corr_df = (pd.DataFrame(corr_results)
             .set_index('Feature')
             .sort_values('Spearman rho', key=abs, ascending=False))
corr_df


## 4. Data Visualization with Matplotlib

### 4.1 Target Distribution

In [ ]:
labels = ['Low (0)', 'Medium (1)', 'High (2)', 'Very High (3)']
counts = df['price_range'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(labels, counts.values, color=PALETTE, edgecolor='white', linewidth=1.2)
ax.set_title('Price Range Distribution (perfectly balanced)', fontweight='bold')
ax.set_ylabel('Count')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            str(int(bar.get_height())), ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.savefig('fig_01_target_distribution.png', bbox_inches='tight')
plt.show()


### 4.2 Histograms – Continuous Feature Distributions

In [ ]:
n = len(continuous_cols)
ncols = 4
nrows = (n + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(16, nrows * 3))
axes = axes.flatten()

for i, col in enumerate(continuous_cols):
    ax = axes[i]
    ax.hist(df[col], bins=30, color='#4C72B0', edgecolor='white', alpha=0.85)
    ax.set_title(col, fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Freq')

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Feature Distributions (Histograms)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig_02_histograms.png', bbox_inches='tight')
plt.show()


### 4.3 Box Plots – Top Features by Price Class

In [ ]:
# Show the 6 most discriminating features (highest ANOVA F-statistic)
top_features = anova_df.head(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, col in enumerate(top_features):
    ax = axes[i]
    data_by_class = [df[df['price_range'] == g][col].values for g in [0,1,2,3]]
    bp = ax.boxplot(data_by_class, patch_artist=True,
                    medianprops={'color': 'black', 'linewidth': 2})
    for patch, color in zip(bp['boxes'], PALETTE):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.set_title(col, fontweight='bold')
    ax.set_xticklabels(['Low', 'Med', 'High', 'V.High'])
    ax.set_xlabel('Price Range')

fig.suptitle('Top-6 Features by Price Class', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_03_boxplots.png', bbox_inches='tight')
plt.show()


### 4.4 Scatter Plot – RAM vs Battery Power

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for cls, color in zip([0,1,2,3], PALETTE):
    sub = df[df['price_range'] == cls]
    ax.scatter(sub['ram'], sub['battery_power'],
               c=color, label=f'Class {cls}', alpha=0.4, s=20, edgecolors='none')

ax.set_xlabel('RAM (MB)')
ax.set_ylabel('Battery Power (mAh)')
ax.set_title('RAM vs Battery Power by Price Range', fontweight='bold')
ax.legend(title='Price Class')
plt.tight_layout()
plt.savefig('fig_04_scatter.png', bbox_inches='tight')
plt.show()


### 4.5 Correlation Heatmap

In [ ]:
corr_matrix = df.corr()
cols = corr_matrix.columns.tolist()

fig, ax = plt.subplots(figsize=(13, 10))
im = ax.imshow(corr_matrix, cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_xticks(range(len(cols)))
ax.set_yticks(range(len(cols)))
ax.set_xticklabels(cols, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(cols, fontsize=9)

for i in range(len(cols)):
    for j in range(len(cols)):
        val = corr_matrix.iloc[i, j]
        color = 'white' if abs(val) > 0.5 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=6.5, color=color)

ax.set_title('Pearson Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_05_heatmap.png', bbox_inches='tight')
plt.show()


### 4.6 Feature Discriminating Power (ANOVA F-statistic)

In [ ]:
sorted_anova = anova_df['F-statistic'].sort_values()
colors_bar = ['#C44E52' if v > 100 else '#4C72B0' for v in sorted_anova.values]

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(sorted_anova.index, sorted_anova.values, color=colors_bar, edgecolor='white')
ax.axvline(100, color='grey', linestyle='--', linewidth=1, label='F = 100')
ax.set_xlabel('F-statistic (ANOVA)')
ax.set_title('Feature Discriminating Power by Price Class', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('fig_06_fstat.png', bbox_inches='tight')
plt.show()


## 5. Insight Synthesis & Conclusion

### Key Findings

| Finding | Detail |
|---------|--------|
| **RAM dominates** | Highest F-statistic by a wide margin; Spearman rho ~0.92 with price class. |
| **Battery power matters** | Second-strongest predictor; higher battery correlates with higher price tier. |
| **Camera & resolution** | `px_height`, `px_width`, `pc` also significantly discriminate classes. |
| **Clock speed is weak** | Low F-statistic; MHz alone is not a reliable price indicator in this dataset. |
| **Binary features help modestly** | `four_g`, `three_g` show moderate differences; `blue` (Bluetooth) is near-random w.r.t. price. |
| **Clean data** | No missing values, no duplicates – no imputation required. |
| **Non-normality** | Shapiro-Wilk rejects normality for all features -> prefer non-parametric or tree-based models. |

### Unexpected Findings
- **`touch_screen`** has virtually no correlation with price – nearly all devices are touch regardless of tier.
- **`n_cores`** shows a surprisingly low F-statistic; more CPU cores does not imply a higher price.
- **`sc_w` & `sc_h`** (screen cm) are less discriminating than pixel resolution.

